# Laboratorio 1 — Series de Tiempo
### Data Science | Universidad del Valle de Guatemala 

**Integrantes:** Adrián González, José Ordoñez, Alejandro Anton

Dataset: ingreso mensual de viajeros internacionales a Guatemala (2009-2026).

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
FIGS = "figs"
os.makedirs(FIGS, exist_ok=True)

## Carga de datos

In [2]:
df = pd.read_excel("Base_Migracion_2009-2026jun.xlsx")
print("Dimensiones:", df.shape)
df.dtypes

Dimensiones: (161036, 13)


Año                        int64
Mes cod                    int64
Mes                          str
Vía                          str
Frontera                     str
País                         str
Región                       str
Región dos                   str
Regiones OMT                 str
MCEO                         str
Agrupación Residencia        str
Tipo de Viajero              str
Viajero                  float64
dtype: object

## Limpieza y calidad de datos

Revisamos nulos, duplicados, valores atípicos y consistencia de categorías.

In [ ]:
print("--- Nulos por columna ---")
print(df.isna().sum())
print("\n--- Duplicados exactos ---", df.duplicated().sum())

# Fecha de referencia (dia 1 de cada mes)
df["Fecha"] = pd.to_datetime(dict(year=df["Año"], month=df["Mes cod"], day=1))

# Región dos: hay valores '0' y separacion redundante
print("\n--- Valores unicos de 'Región dos' ---")
print(df["Región dos"].value_counts())

--- Nulos por columna ---
Año                      0
Mes cod                  0
Mes                      0
Vía                      0
Frontera                 0
País                     0
Región                   0
Región dos               0
Regiones OMT             0
MCEO                     0
Agrupación Residencia    0
Tipo de Viajero          0
Viajero                  0
dtype: int64

--- Duplicados exactos --- 0

--- Valores unicos de 'Región dos' ---
Región dos
Europa                         54640
América Del Centro             31362
América Del Sur y el Caribe    24734
América Del Norte              17306
Asia                           13247
Otros Paises Del Mundo          8979
Oceanía                         5746
Oriente Medio                   4805
Cruceristas                      196
0                                 13
Cruceros                           8
Name: count, dtype: int64


In [4]:
# Unificamos categorias de cruceros dentro de Región dos
df["Región dos"] = df["Región dos"].replace({"Cruceros": "Cruceristas", "0": "Sin Clasificar"})

# Viajero: cantidad de viajeros, se esperan enteros pero hay decimales -> posible
# resultado de agregaciones/estimaciones oficiales. Revisamos negativos y ceros.
print("Registros con Viajero < 0:", (df["Viajero"] < 0).sum())
print("Registros con Viajero == 0:", (df["Viajero"] == 0).sum())
print("Registros con parte decimal no entera:", (df["Viajero"] % 1 != 0).sum())

Registros con Viajero < 0: 0
Registros con Viajero == 0: 54
Registros con parte decimal no entera: 51272


In [5]:
# Outliers univariados (a nivel de registro) via IQR
q1, q3 = df["Viajero"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 3 * iqr
outliers = df[df["Viajero"] > lim_sup]
print(f"Registros por encima de Q3+3*IQR ({lim_sup:.1f}): {len(outliers)} de {len(df)}")
outliers.groupby("Tipo de Viajero")["Viajero"].agg(["count", "mean", "max"])

Registros por encima de Q3+3*IQR (149.6): 21104 de 161036


,count,mean,max
Tipo de Viajero,,,
Cruceristas,203,5439.985222,21680.933573
Excursionista,3991,2201.153453,38463.026064
Turista,13915,2582.886672,83511.000000
Viajero,2995,1366.694064,92336.035067


## Estadísticas descriptivas

In [6]:
print("--- Estadísticas descriptivas de Viajero (nivel registro) ---")
print(df["Viajero"].describe())
print("\n--- Total de viajeros por Tipo de Viajero ---")
df.groupby("Tipo de Viajero")["Viajero"].sum().sort_values(ascending=False)

--- Estadísticas descriptivas de Viajero (nivel registro) ---
count    161036.000000
mean        324.697193
std        2387.745140
min           0.000000
25%           2.000000
50%           7.000000
75%          38.891667
max       92336.035067
Name: Viajero, dtype: float64

--- Total de viajeros por Tipo de Viajero ---


Tipo de Viajero
Turista          3.764273e+07
Excursionista    9.069184e+06
Viajero          4.471622e+06
Cruceristas      1.104402e+06
Name: Viajero, dtype: float64